In [0]:
#| default_exp convert

## Converting Python into notebooks

Move Python modules into nbdev notebooks while preserving an exportable story.

In [0]:
#| export
import ast,copy
from pathlib import Path

from fastcore.nbio import mk_cell, new_nb
from fastcore.nbio import write_nb as _write_nb
from fastcore.script import call_parse
from fastcore.xtras import pglob

from nbskill.foundation import _cli_return, _is_definition_node, _node_start_line, _tracked_call

In [ ]:
#| export
def _node_source(lines, node):
    start = min([node.lineno, *[d.lineno for d in getattr(node, "decorator_list", [])]]) - 1
    return "\n".join(lines[start:node.end_lineno]).strip("\n")

In [ ]:
#| export
def _node_line_count(node):
    start = min([node.lineno, *[d.lineno for d in getattr(node, "decorator_list", [])]])
    return node.end_lineno - start + 1

In [ ]:
#| export
def _patchable_method(node):
    if not isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)): return False
    if node.decorator_list: return False
    return bool(node.args.posonlyargs or node.args.args)

In [ ]:
#| export
def _annotate_first_arg(node, class_name):
    args = node.args.posonlyargs or node.args.args
    if not args: return
    args[0].annotation = ast.Name(id=class_name, ctx=ast.Load())

In [ ]:
#| export
def _patch_method_source(method, class_name):
    node = copy.deepcopy(method)
    node.decorator_list = []
    _annotate_first_arg(node, class_name)
    ast.fix_missing_locations(node)
    return f"@patch\n{ast.unparse(node)}"

In [ ]:
#| export
def _class_without_methods(lines, cls, methods):
    class_start = min([cls.lineno, *[d.lineno for d in cls.decorator_list]])
    class_line = cls.lineno - class_start
    src_lines = lines[class_start - 1:cls.end_lineno]
    remove_ranges = []
    for method in methods:
        start = min([method.lineno, *[d.lineno for d in method.decorator_list]]) - class_start
        stop = method.end_lineno - class_start + 1
        remove_ranges.append((start, stop))
    for start, stop in sorted(remove_ranges, reverse=True): del src_lines[start:stop]
    body = src_lines[class_line + 1:]
    if not any(line.strip() for line in body): src_lines.append("    pass")
    return "\n".join(src_lines).strip("\n")

In [ ]:
#| export
def _export_cell(source):
    return mk_cell(f"#| export\n{source.strip()}")

In [ ]:
#| export
def _py2nb_cells(source, default_exp, class_lines=100, method_lines=10):
    tree = ast.parse(source)
    lines = source.splitlines()
    cells = [mk_cell(f"#| default_exp {default_exp}")]
    pending_imports = []
    needs_patch = False

    def flush_imports():
        if pending_imports:
            cells.append(_export_cell("\n".join(pending_imports)))
            pending_imports.clear()

    for node in tree.body:
        if isinstance(node, (ast.Import, ast.ImportFrom)):
            pending_imports.append(_node_source(lines, node))
            continue
        if isinstance(node, ast.Assign) and any(isinstance(target, ast.Name) and target.id == "__all__" for target in node.targets):
            continue
        if isinstance(node, ast.AnnAssign) and isinstance(node.target, ast.Name) and node.target.id == "__all__":
            continue
        if isinstance(node, ast.AugAssign) and isinstance(node.target, ast.Name) and node.target.id == "__all__":
            continue
        flush_imports()
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)): cells.append(_export_cell(_node_source(lines, node)))
        elif isinstance(node, ast.ClassDef):
            methods = [child for child in node.body if _patchable_method(child) and _node_line_count(child) > method_lines]
            if _node_line_count(node) > class_lines and methods:
                needs_patch = True
                cells.append(_export_cell(_class_without_methods(lines, node, methods)))
                for method in methods: cells.append(_export_cell(_patch_method_source(method, node.name)))
            else: cells.append(_export_cell(_node_source(lines, node)))
        elif not (isinstance(node, ast.Expr) and isinstance(node.value, ast.Constant) and isinstance(node.value.value, str)):
            cells.append(_export_cell(_node_source(lines, node)))
    flush_imports()
    if needs_patch: cells.insert(1, _export_cell("from fastcore.basics import patch"))
    return cells

In [ ]:
#| export
def _py2nb_file(path, nbs_path="nbs", dest=None, class_lines=100, method_lines=10):
    pth = Path(path)
    source = pth.read_text(encoding="utf-8")
    default_exp = pth.stem
    out_path = Path(dest) if dest else Path(nbs_path) / f"{default_exp}.ipynb"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    nb = new_nb(_py2nb_cells(source, default_exp, class_lines=class_lines, method_lines=method_lines))
    _write_nb(nb, out_path)
    return out_path, len(nb.cells)

In [ ]:
#| export
@call_parse
@_tracked_call
def py2nb(
    path: str,  # Python file path
    nbs_path: str = "nbs",  # Folder for generated notebooks
    dest: str | None = None,  # Explicit notebook path; overrides nbs_path
    class_lines: int = 100,  # Split methods out of classes longer than this
    method_lines: int = 10,  # Split methods longer than this out of large classes
):
    "Convert a Python file into an nbdev-style notebook using AST parsing."
    out_path, n_cells = _py2nb_file(path, nbs_path=nbs_path, dest=dest, class_lines=class_lines, method_lines=method_lines)
    msg = f"Wrote {n_cells} cells to {out_path}"
    print(msg)
    return _cli_return(out_path)

In [ ]:
import tempfile as _tempfile
from pathlib import Path as _Path
from fastcore.nbio import read_nb as _read_nb
from nbskill.convert import py2nb

with _tempfile.TemporaryDirectory() as td:
    src = _Path(td) / "demo.py"
    nbs_path = _Path(td) / "nbs"
    src.write_text("def add(a, b):\n    return a + b\n", encoding="utf-8")
    py2nb(str(src), nbs_path=str(nbs_path))
    nb = _read_nb(nbs_path / "demo.ipynb")
    assert any("#| default_exp demo" in cell.source for cell in nb.cells)
    assert any("def add" in cell.source for cell in nb.cells)

In [ ]:
#| export
@call_parse
@_tracked_call
def py2nbs(
    path: str,  # Folder containing Python files
    nbs_path: str = "nbs",  # Folder for generated notebooks
    recursive: bool = True,  # Search subfolders
    maxdepth: int | None = None,  # Maximum folder depth to search
    preserve_tree: bool = True,  # Preserve folder structure below nbs_path
    class_lines: int = 100,  # Split methods out of classes longer than this
    method_lines: int = 10,  # Split methods longer than this out of large classes
):
    "Convert all Python files in a folder into nbdev-style notebooks."
    root = Path(path)
    py_files = pglob(root, exts="py", recursive=recursive, maxdepth=maxdepth)
    outs = []
    for pth in py_files:
        dest = None
        if preserve_tree and root.is_dir():
            dest = Path(nbs_path) / pth.relative_to(root).with_suffix(".ipynb")
        out_path, n_cells = _py2nb_file(str(pth), nbs_path=nbs_path, dest=str(dest) if dest else None,
                                        class_lines=class_lines, method_lines=method_lines)
        outs.append(out_path)
        print(f"Wrote {n_cells} cells to {out_path}")
    print(f"Converted {len(outs)} Python files to {nbs_path}")
    return _cli_return(outs)